# 🥈 Silver Layer - Spatial + LLM Enrichment (Shift-Left AI)

This notebook performs:

1. **SPATIAL STANDARDIZATION** (via Apache Sedona):
   - Convert Lat/Lng columns to ST_Point geometries
   - Parse GeoJSON polygons for neighborhoods
   - Standardize to EPSG:4326

2. **AI ENRICHMENT (Shift-Left Pattern)**:
   - PySpark Pandas UDF calling local Ollama
   - Process text descriptions from US Accidents & NYC 311
   - Output STRICT JSON: {"severity": 1-10, "hazard_type": "string"}

## Key Pattern
**ALL LLM inference happens in Silver layer**
The Gold layer will ONLY do dimensional modeling and spatial joins.

In [ ]:
# Import transformation functions from src module
import sys
sys.path.append('../src')

from silver_enrichment import (
    Config,
    create_spark_session,
    create_geometry_from_latlon,
    parse_geojson_geometry,
    create_ollama_enrichment_udf,
    parse_ai_enrichment,
    read_bronze_table,
    transform_us_accidents,
    transform_nyc_311,
    transform_usgs_earthquakes,
    transform_osm_infrastructure,
    transform_us_neighborhoods,
    write_silver_table,
    run_silver_enrichment
)

print("✅ Imports successful")

## Configuration

In [ ]:
# Display current configuration
config = Config()

print("=" * 60)
print("SILVER LAYER CONFIGURATION")
print("=" * 60)
print(f"App Name: {config.APP_NAME}")
print(f"Spark Master: {config.SPARK_MASTER}")
print(f"Bronze Bucket: {config.BRONZE_BUCKET}")
print(f"Silver Bucket: {config.SILVER_BUCKET}")
print(f"Ollama URL: {config.OLLAMA_BASE_URL}")
print(f"Ollama Model: {config.OLLAMA_MODEL}")
print(f"Local Data Dir: {config.DELTA_COMPRESSION}")

## Create Spark Session

In [ ]:
# Create Spark session
spark = create_spark_session(config)
print(f"✅ Spark session created: {spark.version}")

## Step 1: Test Geometry Creation (Sedona)

In [ ]:
# Cell disabled for basic testing

## Step 2: Test GeoJSON Parsing

In [ ]:
# Cell disabled for basic testing

## Step 3: Test LLM Enrichment (Ollama)

In [ ]:
# Test Ollama enrichment with mock (for testing without running Ollama)
import os
os.environ['OLLAMA_BASE_URL'] = 'http://localhost:11434'
os.environ['OLLAMA_MODEL'] = 'llama3.2:1b'

# Test parsing AI enrichment
test_json = '{"severity": 7, "hazard_type": "traffic_collision"}'
result = parse_ai_enrichment(test_json)
print(f"✅ Parse result: {result}")

# Test bounds checking
bad_json = '{"severity": 15, "hazard_type": "test"}'
result_bad = parse_ai_enrichment(bad_json)
print(f"✅ Bounds check (invalid severity): {result_bad}")

## Step 4: Transform US Accidents (with LLM)

In [ ]:
# Transform US Accidents
# Note: This step would call Ollama for real LLM enrichment
# For testing, we use mock data

# Create sample data
from pyspark.sql import Row

data = [
    Row(incident_id="US-001", incident_description="Car accident on highway", latitude=40.7128, longitude=-74.0060),
    Row(incident_id="US-002", incident_description="Pedestrian hit while crossing", latitude=40.7589, longitude=-73.9851),
]

df_sample = spark.createDataFrame(data)

# Show sample
print("📊 Sample US Accidents:")
df_sample.show()

## Step 5: Write Silver Tables

In [ ]:
# Write silver tables (using sample data for demonstration)
# In production, run: write_silver_table(df, 'us_accidents', 'overwrite')

output_dir = "/tmp/geoai/silver"
df_sample.write.format("parquet").mode("overwrite").save(f"{output_dir}/us_accidents")
print(f"✅ Wrote us_accidents to {output_dir}/us_accidents")

# Verify
verify_df = spark.read.format("parquet").load(f"{output_dir}/us_accidents")
verify_df.show()

## Silver Layer Summary

In [ ]:
# Summary
print("=" * 60)
print("SILVER LAYER SUMMARY")
print("=" * 60)
print()
print("✅ Functions Available:")
print("  - create_spark_session()")
print("  - create_geometry_from_latlon()")
print("  - parse_geojson_geometry()")
print("  - create_ollama_enrichment_udf()")
print("  - parse_ai_enrichment()")
print("  - read_bronze_table()")
print("  - transform_us_accidents()")
print("  - transform_nyc_311()")
print("  - transform_usgs_earthquakes()")
print("  - transform_osm_infrastructure()")
print("  - transform_us_neighborhoods()")
print("  - write_silver_table()")
print()
print("📁 Output: /tmp/geoai/silver/")

---

## Next Steps

Proceed to **Gold Layer** for:
- Kimball Star Schema dimensional modeling
- Spatial joins (events to neighborhoods, events to nearest infrastructure)
- NO LLM calls (pure declarative SQL)
- Aggregated metrics and analytics